# LifeMate baseline — Qwen3-4B zero-shot on gold transcripts

1. Locally: `python pack.py` → creates `dist/lifemate_code.zip` and `dist/zaic_data_noaudio.zip`.
2. Upload both zips to Google Drive `MyDrive/zaloai/` (keep them private: the organiser data may not be redistributed).
3. Runtime → GPU (a T4 is enough; L4/A100 are faster). Run the cells in order.

On Kaggle: upload both zips as one private Dataset and set `SRC` in cell 2 to `/kaggle/input/<dataset-name>`.

In [ ]:
!nvidia-smi

In [ ]:
import os, zipfile
try:
    from google.colab import drive
    drive.mount('/content/drive')
    SRC = '/content/drive/MyDrive/zaloai'
    WORK = '/content/work'
except ImportError:  # Kaggle
    SRC = '/kaggle/input/zaloai'
    WORK = '/kaggle/working/work'
os.makedirs(WORK + '/data', exist_ok=True)
zipfile.ZipFile(f'{SRC}/lifemate_code.zip').extractall(WORK)
zipfile.ZipFile(f'{SRC}/zaic_data_noaudio.zip').extractall(WORK + '/data')
os.chdir(WORK)
os.environ['LIFEMATE_DATA'] = WORK + '/data'
print(os.listdir(WORK), os.listdir(WORK + '/data'))

In [ ]:
!pip install -q vllm

In [ ]:
# sanity: oracle must print SCORE 1.0000
!python -m lifemate.run --split train --ids val_seen --backend oracle

In [ ]:
MODEL = 'Qwen/Qwen3-4B-Instruct-2507'
!python -m lifemate.run --split train --ids val_seen --backend vllm --model {MODEL} --out runs/val_seen_qwen3_4b

In [ ]:
!python -m lifemate.run --split train --ids val_unseen --backend vllm --model {MODEL} --out runs/val_unseen_qwen3_4b

In [ ]:
# quick error analysis: zero-score samples by group
import json, collections
for run in ['runs/val_seen_qwen3_4b', 'runs/val_unseen_qwen3_4b']:
    rows = [json.loads(l) for l in open(f'{run}/scores.jsonl')]
    print(run, collections.Counter((r['category'], 'hash_ok' if r['hash_ok'] else 'hash_fail') for r in rows if r['score'] == 0))

In [ ]:
# save the results to Drive / output
import shutil
shutil.make_archive(f'{SRC if SRC.startswith("/content") else "/kaggle/working"}/runs_baseline', 'zip', 'runs')